# 220119 F2 run11: Cluster, Clean, and Reconstruct

Stage 2 for the restarted V2a workflow. This notebook loads the saved decomposition artifacts from `decompositions.ipynb`, clusters IC spectra, ranks clusters in the configured low-frequency band, reconstructs cleaned traces from accepted clusters, and saves auditable cluster selections plus descriptive trace-preservation metrics.

This full-recording clustering is for exploration, visualization, and candidate generation. Use `behavior_decoding.ipynb` for leakage-safe fold-local behavior and causal-state evidence.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists() and (path / "src").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option("display.max_colwidth", 120)
print(f"Project root: {PROJECT_ROOT}")


from ica_denoising.behavior_decoding import TraceVariant, summarize_trace_preservation
from ica_denoising.bss_notebook import (
    BSS_METHODS,
    cleaned_trace_output_paths,
    load_bss_decomposition_outputs,
    load_traces,
    save_cleaned_trace_output,
    save_cluster_selection_output,
)
from ica_denoising.core.ica_utils import (
    cluster,
    plot_mean_log_psd_by_cluster,
    rank_clusters_by_mean_log_psd,
    reconstruct_bss,
)
from ica_denoising.core.visualization import plot_clusters


In [ ]:
DATASET_KEY = "v2a-RSNs/220119_F2_run11_fluorescence"
METHODS_TO_RUN = list(BSS_METHODS)
N_CLUSTERS = 7
FEATURE_START_BIN = 30
WELCH_NPERSEG = 250
WELCH_NOVERLAP = 125
RANKING_FMIN_HZ = 0.0
RANKING_FMAX_HZ = 0.20
RANKING_AGGREGATE = "peak"
KEEP_TOP_CLUSTER_COUNT = 2
RANDOM_STATE = 0
PLOT_DPI = 300
SAVE_OUTPUTS = True


In [ ]:
dataset, traces = load_traces(DATASET_KEY, PROJECT_ROOT)
sample_rate_hz = float(dataset.sample_rate_hz)
results = {}
skipped = {}
for method in METHODS_TO_RUN:
    try:
        results[method] = load_bss_decomposition_outputs(DATASET_KEY, method, PROJECT_ROOT)
    except FileNotFoundError as error:
        skipped[method] = str(error).splitlines()[0]

if not results:
    raise FileNotFoundError("No saved decompositions found. Run decompositions.ipynb first.")
dataset_output_dir = next(iter(results.values())).output_dir.parent
figure_dir = dataset_output_dir / "figures"
if SAVE_OUTPUTS:
    figure_dir.mkdir(parents=True, exist_ok=True)
print(f"Dataset: {dataset.key}")
print(f"Loaded methods: {list(results)}")
print(f"Skipped methods: {skipped}")
print(f"Dataset output directory: {dataset_output_dir.relative_to(PROJECT_ROOT)}")


In [ ]:
def cluster_detail_records(labels, ranking, accepted_clusters):
    labels = np.asarray(labels)
    accepted_set = {int(cluster_id) for cluster_id in accepted_clusters}
    ranking_by_cluster = {int(row["cluster"]): row for row in ranking}
    cluster_records = []
    component_records = []
    for label in sorted(np.unique(labels).astype(int).tolist()):
        component_indices = np.flatnonzero(labels == label).astype(int).tolist()
        rank_row = ranking_by_cluster.get(label, {})
        rank = rank_row.get("rank")
        score = rank_row.get("score")
        accepted = label in accepted_set
        cluster_records.append(
            {
                "cluster": int(label),
                "rank": None if rank is None else int(rank),
                "score": None if score is None else float(score),
                "n_components": len(component_indices),
                "accepted": bool(accepted),
                "component_indices": component_indices,
            }
        )
        for component in component_indices:
            component_records.append(
                {
                    "component": int(component),
                    "cluster": int(label),
                    "cluster_rank": None if rank is None else int(rank),
                    "cluster_score": None if score is None else float(score),
                    "accepted": bool(accepted),
                }
            )
    return cluster_records, component_records


cluster_results = {}
ranking_rows = []
for method, result in results.items():
    embedding, labels, spectra, features = cluster(
        result.ic_comps,
        N_CLUSTERS,
        sample_rate_hz,
        feature_start_bin=FEATURE_START_BIN,
        nperseg=WELCH_NPERSEG,
        noverlap=WELCH_NOVERLAP,
        random_state=RANDOM_STATE,
    )
    ranking = rank_clusters_by_mean_log_psd(
        spectra,
        labels,
        sample_rate_hz,
        fmin=RANKING_FMIN_HZ,
        fmax=RANKING_FMAX_HZ,
        aggregate=RANKING_AGGREGATE,
    )
    cluster_results[method] = {
        "embedding": embedding,
        "labels": labels,
        "spectra": spectra,
        "features": features,
        "ranking": ranking,
    }
    ranking_rows.extend({"method": method, **row} for row in ranking)

cluster_ranking = pd.DataFrame(ranking_rows)
cluster_ranking_path = dataset_output_dir / "cluster_ranking.csv"
if SAVE_OUTPUTS:
    cluster_ranking.to_csv(cluster_ranking_path, index=False)
    print(f"Saved cluster ranking: {cluster_ranking_path.relative_to(PROJECT_ROOT)}")
cluster_ranking


In [ ]:
figure_paths = {}
for method, clustered in cluster_results.items():
    method_cluster_dir = results[method].output_dir / "clusters"
    if SAVE_OUTPUTS:
        method_cluster_dir.mkdir(parents=True, exist_ok=True)

    cluster_fig, _, _ = plot_clusters(clustered["embedding"], clustered["labels"])
    cluster_fig.suptitle(f"{dataset.data_name}: {method} IC spectral clusters", y=1.02)
    cluster_fig.tight_layout()
    cluster_path = method_cluster_dir / f"ic_clusters_{method}_{dataset.data_name}.png"
    manuscript_cluster_path = figure_dir / f"cluster_embedding_{method}_{dataset.data_name}.png"

    psd_fig, _ = plot_mean_log_psd_by_cluster(
        clustered["spectra"],
        clustered["labels"],
        sample_rate_hz,
        xlim=(0.0, min(0.75, sample_rate_hz / 2.0)),
        title_prefix=f"{method} cluster",
    )
    psd_fig.suptitle(f"{dataset.data_name}: {method} mean log PSD", y=1.02)
    psd_path = method_cluster_dir / f"mean_log_psd_{method}_{dataset.data_name}.png"
    manuscript_psd_path = figure_dir / f"cluster_mean_log_psd_{method}_{dataset.data_name}.png"

    if SAVE_OUTPUTS:
        cluster_fig.savefig(cluster_path, dpi=PLOT_DPI, bbox_inches="tight")
        cluster_fig.savefig(manuscript_cluster_path, dpi=PLOT_DPI, bbox_inches="tight")
        psd_fig.savefig(psd_path, dpi=PLOT_DPI, bbox_inches="tight")
        psd_fig.savefig(manuscript_psd_path, dpi=PLOT_DPI, bbox_inches="tight")
        print(f"Saved cluster plots for {method}: {method_cluster_dir.relative_to(PROJECT_ROOT)}")

    figure_paths[method] = {
        "cluster_embedding": cluster_path,
        "cluster_embedding_manuscript": manuscript_cluster_path,
        "mean_log_psd": psd_path,
        "mean_log_psd_manuscript": manuscript_psd_path,
    }
    plt.show()


In [ ]:
cleaned_by_method = {}
selection_rows = []
saved_paths = {}
for method, result in results.items():
    clustered = cluster_results[method]
    accepted_clusters = [
        int(row["cluster"]) for row in clustered["ranking"][:KEEP_TOP_CLUSTER_COUNT]
    ]
    rejected_clusters = sorted(set(np.unique(clustered["labels"]).astype(int)) - set(accepted_clusters))
    accepted_components = np.flatnonzero(np.isin(clustered["labels"], accepted_clusters)).astype(int)
    rejected_components = np.flatnonzero(~np.isin(clustered["labels"], accepted_clusters)).astype(int)
    cleaned = reconstruct_bss(result.ic_comps, result.A, result.mean, keep=accepted_components)
    cleaned_by_method[method] = cleaned
    cluster_records, component_records = cluster_detail_records(
        clustered["labels"],
        clustered["ranking"],
        accepted_clusters,
    )
    planned_cleaned_paths = cleaned_trace_output_paths(dataset, method, result.output_dir)
    selection_payload = {
        "selection_scope": "full_recording_descriptive",
        "n_clusters_requested": int(N_CLUSTERS),
        "n_clusters_observed": int(np.unique(clustered["labels"]).size),
        "feature_start_bin": int(FEATURE_START_BIN),
        "welch_nperseg": int(WELCH_NPERSEG),
        "welch_noverlap": int(WELCH_NOVERLAP),
        "ranking_band_hz": [float(RANKING_FMIN_HZ), float(RANKING_FMAX_HZ)],
        "ranking_aggregate": RANKING_AGGREGATE,
        "keep_top_cluster_count": int(KEEP_TOP_CLUSTER_COUNT),
        "accepted_clusters": accepted_clusters,
        "rejected_clusters": [int(value) for value in rejected_clusters],
        "accepted_components": accepted_components.tolist(),
        "rejected_components": rejected_components.tolist(),
        "cluster_ranking": clustered["ranking"],
        "clusters": cluster_records,
        "components": component_records,
        "decomposition_paths": {
            label: str(path.relative_to(PROJECT_ROOT)) for label, path in result.saved_paths.items()
        },
        "cleaned_output_paths": {
            label: str(path.relative_to(PROJECT_ROOT)) for label, path in planned_cleaned_paths.items()
        },
        "figure_paths": {
            label: str(path.relative_to(PROJECT_ROOT)) for label, path in figure_paths.get(method, {}).items()
        },
    }
    selection_path = None
    cleaned_paths = {}
    if SAVE_OUTPUTS:
        selection_path = save_cluster_selection_output(
            spec=dataset,
            method=method,
            output_dir=result.output_dir,
            selection=selection_payload,
        )
        cleaned_paths = save_cleaned_trace_output(
            spec=dataset,
            method=method,
            traces=traces,
            cleaned=cleaned,
            output_dir=result.output_dir,
            reject_components=rejected_components,
            metadata={
                "selection_scope": "full_recording_descriptive",
                "cluster_selection_path": str(selection_path.relative_to(PROJECT_ROOT)),
                "accepted_clusters": accepted_clusters,
                "accepted_components": accepted_components.tolist(),
            },
        )
        saved_paths[method] = {"selection": selection_path, **cleaned_paths}
    selection_rows.append(
        {
            "method": method,
            "accepted_clusters": accepted_clusters,
            "accepted_component_count": int(len(accepted_components)),
            "rejected_component_count": int(len(rejected_components)),
            "output_dir": str(result.output_dir.relative_to(PROJECT_ROOT)),
            "cluster_selection_path": None if selection_path is None else str(selection_path.relative_to(PROJECT_ROOT)),
            "cleaned_path": None if not cleaned_paths else str(cleaned_paths["cleaned"].relative_to(PROJECT_ROOT)),
        }
    )

selection_summary = pd.DataFrame(selection_rows)
selection_summary_path = dataset_output_dir / "cluster_selection_summary.csv"
if SAVE_OUTPUTS:
    selection_summary.to_csv(selection_summary_path, index=False)
    print(f"Saved cluster selection summary: {selection_summary_path.relative_to(PROJECT_ROOT)}")
selection_summary


In [ ]:
trace_variants = [TraceVariant("raw", dataset.trace_path, traces.T)] + [
    TraceVariant(method, saved_paths.get(method, {}).get("cleaned", Path(method)), cleaned)
    for method, cleaned in cleaned_by_method.items()
]
trace_preservation = summarize_trace_preservation(trace_variants, reference_name="raw")
trace_metrics_path = dataset_output_dir / "clustering_trace_preservation.csv"
if SAVE_OUTPUTS:
    trace_preservation.to_csv(trace_metrics_path, index=False)
print(f"Trace metrics: {trace_metrics_path.relative_to(PROJECT_ROOT)}")
trace_preservation


In [ ]:
fig, axes = plt.subplots(len(cleaned_by_method), 1, figsize=(15, 2.8 * len(cleaned_by_method)), sharex=True)
axes = np.atleast_1d(axes)
neuron = 0
stop = min(700, traces.shape[1])
for ax, (method, cleaned) in zip(axes, cleaned_by_method.items()):
    ax.plot(traces[neuron, :stop], label="raw", color="black", alpha=0.6, lw=0.8)
    ax.plot(cleaned[:stop, neuron], label=method, lw=0.8)
    ax.set_title(method)
    ax.legend(loc="upper right")
axes[-1].set_xlabel("Frame")
fig.tight_layout()
trace_comparison_path = figure_dir / f"cluster_cleaned_trace_comparison_{dataset.data_name}.png"
if SAVE_OUTPUTS:
    fig.savefig(trace_comparison_path, dpi=PLOT_DPI, bbox_inches="tight")
    print(f"Saved trace comparison plot: {trace_comparison_path.relative_to(PROJECT_ROOT)}")
